<a href="https://colab.research.google.com/github/jpetr019/STAT108Project/blob/main/Joshua_Petrikat_STAT_CS108_S26_Course_Project_Bias_Mitigation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STAT/CS108 S26 Course Project

### Team Details

- Teammate 1: Joshua Petrikat
- Teammate 2: Jason Margulies
- Teammate 3: Alejandro Leon

---

# Installs

In [4]:
# [INSERT CODE HERE to install necessary packages]
!pip install ucimlrepo
!pip install fairlearn
!pip install seaborn matplotlib numpy pandas scikit-learn ucimlrepo


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: C:\Users\leona\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: C:\Users\leona\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: C:\Users\leona\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Imports

In [5]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import collections
from pprint import pprint
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score

## Add additional imports needed for your project here.

from ucimlrepo import fetch_ucirepo

# Loading dataset
_(same as previous milestone, copy-paste)_

In [6]:
# Load your selected dataset

# fetch dataset
adult = fetch_ucirepo(id=2)

# data (as pandas dataframes)
X = adult.data.features
y = adult.data.targets

sensitive_feature_colname = ['sex', 'race'] # sensitive feature name

# Make sensitive features-based group labels
group_labels = (X['sex'].astype(str) + '_' + X['race'].astype(str))

# Print some stats
print(f"No. of samples: {X.shape[0]}")
print(f"No. of features: {X.shape[1]}")
print(f"Group Counts: {dict(collections.Counter(group_labels))}")

No. of samples: 48842
No. of features: 14
Group Counts: {'Male_White': 28735, 'Male_Black': 2377, 'Female_Black': 2308, 'Female_White': 13027, 'Male_Asian-Pac-Islander': 1002, 'Male_Amer-Indian-Eskimo': 285, 'Female_Other': 155, 'Female_Asian-Pac-Islander': 517, 'Female_Amer-Indian-Eskimo': 185, 'Male_Other': 251}


# Preparing dataset
_(same as previous milestone, copy-paste)_

In [7]:
# Some subset of following dataset preparation steps may be necessary depending on your dataset,
# 1. Drop unnecessary features
# 2. Handle missing data
# 3. Encode categorical features
# 4. Normalize numerical features
# 5. Encode target (if your task is classification)

# [INSERT YOUR CODE HERE]
# creating copies

X = X.copy()
y = y.squeeze().copy()

# replacing missing value markers with NaN
X = X.replace('?', np.nan)

# dropping rows with missing values
mask = X.notna().all(axis=1) & y.notna()
X = X.loc[mask]
y = y.loc[mask]
group_labels = group_labels.loc[mask]

# encoding target labels (salary) as 0/1

y = y.astype(str).str.strip().str.replace('.', '', regex=False)
y = y.map({'<=50K': 0, '>50K': 1})

# one hot encoding for categorical features since logisitc regression requires numerical values
X = pd.get_dummies(X, drop_first=True) # adding drop_first to avoid redundant columns with sex_Female: 0 and sex_Male: 1 that communicate the same thing

# Note: X and y have been modified before the following lines of code!
print(f"No. of samples AFTER cleaning: {X.shape[0]}")
assert X.shape[0] == y.shape[0] == group_labels.shape[0] ## Ensure that the target and group_labels have been updated if some samples were removed during cleaning.
print(f"No. of features AFTER encoding: {X.shape[1]}")

No. of samples AFTER cleaning: 45222
No. of features AFTER encoding: 96


# Getting training and testing sets

Note: Train-test split is made **ONCE** to obtain the _training set_ and the _testing set_ and every teammate will use the training set to train their baseline model and test the trained model using the testing set. **NEVER** modify the testing set once it has been created.
Therefore, the following code cell does not need to be edited.

_(same as previous milestone, copy-paste)_

In [8]:
X_train, X_test, \
  y_train, y_test, \
    group_labels_train, group_labels_test = train_test_split(X, y, group_labels,
                                                             test_size=0.2, random_state=42)

print(f"No. of training samples: {X_train.shape[0]}")
print(f"No. of testing samples: {X_test.shape[0]}")

# Delete X, y and group_label variables to make sure they are not used later on.
del X
del y
del group_labels

No. of training samples: 36177
No. of testing samples: 9045


# Setting up evaluation metrics
Note: The same evaluation function will be used by all teammates.

_(same as previous milestone, copy-paste)_

In [9]:
from sklearn.metrics import accuracy_score, f1_score
from fairlearn.metrics import (
    MetricFrame,
    selection_rate,
    true_positive_rate,
    false_positive_rate,
    demographic_parity_difference,
    demographic_parity_ratio
)

def evaluate_model(y_test, y_pred, g_labels):
  """
  Evaluate the performance of your trained model on the testing set.

  Parameters
  ----------
  y_test : array-like
    The true targets of the testing set.
  y_pred : array-like
    The predicted targets of the testing set.
  g_labels : array-like
    The group labels of the testing set.

  Returns
  -------
  results : dict
    A dictionary containing the evaluation results.

    Example:
      For classification task, the task-specific performance metrics like {'accuracy': <value>, 'f1_score': <value>, ...}
      and fairness metrics like {'demographic_parity': <value>, 'equalized_odds': <value>, ...}.

  """
  results = {}

  # Note: These metrics will be calculated for - 1. the full testing set, 2. individual groups.
  # Task-specific performance metrics
  # [INSERT CODE HERE for performance metrics appropriate for your task]

  # ----------------------------
  # Overall performance metrics
  # ----------------------------
  results["overall"] = {
      "accuracy": accuracy_score(y_test, y_pred),
      "f1_score": f1_score(y_test, y_pred, zero_division=0)
  }

  # ----------------------------
  # Group-level metrics
  # ----------------------------
  metric_frame = MetricFrame(
      metrics={
          "accuracy": accuracy_score,
          "f1_score": lambda y_true, y_pred: f1_score(y_true, y_pred, zero_division=0),
          "selection_rate": selection_rate,
          "true_positive_rate": true_positive_rate,
          "false_positive_rate": false_positive_rate
      },
      y_true=y_test,
      y_pred=y_pred,
      sensitive_features=g_labels
  )

  results["by_group"] = metric_frame.by_group

  # Fairness metric
  # [INSERT CODE HERE for fairness metric appropriate for your task]

  # ----------------------------
  # Fairness metrics
  # ----------------------------

  # Demographic parity:
  # difference in positive prediction rates across groups
  results["demographic_parity_difference"] = demographic_parity_difference(
      y_test,
      y_pred,
      sensitive_features=g_labels
  )

  # Disparate impact ratio:
  # ratio of lowest selection rate to highest selection rate
  results["disparate_impact_ratio"] = demographic_parity_ratio(
      y_test,
      y_pred,
      sensitive_features=g_labels
  )

  # Equal opportunity:
  # difference in true positive rates across groups
  results["equal_opportunity_difference"] = metric_frame.difference()["true_positive_rate"]

  # Equalized odds:
  # max of TPR difference and FPR difference
  tpr_diff = metric_frame.difference()["true_positive_rate"]
  fpr_diff = metric_frame.difference()["false_positive_rate"]

  results["equalized_odds_difference"] = max(tpr_diff, fpr_diff)


  return results

# Training baseline models (INDIVIDUAL CONTRIBUTION)

In [17]:
## A place to save all teammates's baseline results
all_baseline_results = [] ## DO NOT EDIT

## Teammate 1

In [18]:
# Select a model and train it on the training set
# [INSERT YOUR CODE HERE]
model = LogisticRegression(max_iter=1000)
# Make predictions on the testing set and store them in y_pred
model.fit(X_train, y_train)
y_pred = model.predict(X_test) # [INSERT CODE HERE]

# Evaluate testing set predictions using evaluate_model()
results = evaluate_model(y_test, y_pred, group_labels_test)

# Save your results to all_baseline_results
results['teammate'] = 'Teammate 1'
results['experiment_type'] = 'baseline'
results['predictor_model'] = 'Logistic Regression' #[INSERT MODEL NAME HERE]
results['mitigation_strategy'] = 'NONE' ## DO NOT EDIT: This is pre-mitigation baseline
all_baseline_results.append(results)

pprint(results)

C:\Users\leona\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


{'by_group':                            accuracy  f1_score  selection_rate  \
sensitive_feature_0                                             
Female_Amer-Indian-Eskimo  0.900000  0.750000        0.200000   
Female_Asian-Pac-Islander  0.853333  0.421053        0.160000   
Female_Black               0.944321  0.528302        0.055679   
Female_Other               1.000000  1.000000        0.090909   
Female_White               0.912646  0.603774        0.093594   
Male_Amer-Indian-Eskimo    0.844828  0.571429        0.189655   
Male_Asian-Pac-Islander    0.722826  0.662252        0.467391   
Male_Black                 0.864055  0.575540        0.129032   
Male_Other                 0.880952  0.736842        0.214286   
Male_White                 0.804572  0.670458        0.259134   

                           true_positive_rate  false_positive_rate  
sensitive_feature_0                                                 
Female_Amer-Indian-Eskimo            0.750000             0.062500  

## Teammate 2

In [19]:
from sklearn.ensemble import RandomForestClassifier
# Select a model and train it on the training set
# [INSERT YOUR CODE HERE]
model_t2 = RandomForestClassifier(n_estimators=100, random_state=42)
model_t2.fit(X_train, y_train)

# Make predictions on the testing set and store them in y_pred
y_pred = model_t2.predict(X_test)

# Evaluate testing set predictions using evaluate_model()
results = evaluate_model(y_test, y_pred, group_labels_test)

# Save your results to all_baseline_results
results['teammate'] = 'Teammate 2'
results['experiment_type'] = 'baseline'
results['predictor_model'] = 'Random Forest'
results['mitigation_strategy'] = 'NONE' ## DO NOT EDIT: This is pre-mitigation baseline
all_baseline_results.append(results)

pprint(results)

{'by_group':                            accuracy  f1_score  selection_rate  \
sensitive_feature_0                                             
Female_Amer-Indian-Eskimo  0.925000  0.769231        0.125000   
Female_Asian-Pac-Islander  0.906667  0.533333        0.106667   
Female_Black               0.955457  0.600000        0.048998   
Female_Other               1.000000  1.000000        0.090909   
Female_White               0.923877  0.669078        0.103161   
Male_Amer-Indian-Eskimo    0.879310  0.588235        0.120690   
Male_Asian-Pac-Islander    0.788043  0.697674        0.347826   
Male_Black                 0.889401  0.661972        0.135945   
Male_Other                 0.809524  0.428571        0.095238   
Male_White                 0.812254  0.695626        0.282930   

                           true_positive_rate  false_positive_rate  
sensitive_feature_0                                                 
Female_Amer-Indian-Eskimo            0.625000             0.000000  

## Teammate 3

In [20]:
from sklearn.ensemble import GradientBoostingClassifier

# Select a model and train it on the training set
model_t3 = GradientBoostingClassifier(n_estimators=100, random_state=42)
model_t3.fit(X_train, y_train)

# Make predictions on the testing set and store them in y_pred
y_pred = model_t3.predict(X_test)

# Evaluate testing set predictions using evaluate_model()
results = evaluate_model(y_test, y_pred, group_labels_test)

# Save your results to all_baseline_results
results['teammate'] = 'Teammate 3'
results['experiment_type'] = 'baseline'
results['predictor_model'] = 'Gradient Boosting'
results['mitigation_strategy'] = 'NONE' ## DO NOT EDIT: This is pre-mitigation baseline
all_baseline_results.append(results)

pprint(results)

{'by_group':                            accuracy  f1_score  selection_rate  \
sensitive_feature_0                                             
Female_Amer-Indian-Eskimo  0.925000  0.769231        0.125000   
Female_Asian-Pac-Islander  0.920000  0.571429        0.093333   
Female_Black               0.955457  0.565217        0.040089   
Female_Other               1.000000  1.000000        0.090909   
Female_White               0.932196  0.684720        0.088186   
Male_Amer-Indian-Eskimo    0.862069  0.555556        0.137931   
Male_Asian-Pac-Islander    0.782609  0.677419        0.320652   
Male_Black                 0.887097  0.631579        0.115207   
Male_Other                 0.833333  0.461538        0.071429   
Male_White                 0.824246  0.700702        0.253326   

                           true_positive_rate  false_positive_rate  
sensitive_feature_0                                                 
Female_Amer-Indian-Eskimo            0.625000             0.000000  

# Mitigating Bias (INDIVIDUAL CONTRIBUTION)



In [21]:
## A place to save all teammates' post-mitigation results
all_mitigated_results = [] ## DO NOT EDIT

## Teammate 1

In [22]:
# Implement your bias mitigation strategy
## If you chose preprocessing, you will train a new version of your predictor model with new/modified inputs.
## If you chose inprocessing, you will train a new version of your predictor with modified learning objective (loss function).
## If you chose postprocessing, you will implement strategies to modify the predictions (y_pred) of the trained baseline predictor model from the previous milestone without training any new version of the predictor model.

# [INSERT CODE HERE]

sensitive_cols = [
    col for col in X_train.columns
    if col.startswith('sex_') or col.startswith('race_') or col in ['sex', 'race', 'gender']
]

print('Dropping sensitive columns:', sensitive_cols)

X_train_mitigated = X_train.drop(columns = sensitive_cols)
X_test_mitigated = X_test.drop(columns = sensitive_cols)

model_mitigated = LogisticRegression(max_iter=1000)
# Make predictions on the testing set and store them in y_pred
model_mitigated.fit(X_train_mitigated, y_train)

# Make predictions on the testing set and store them in y_pred_mitigate
y_pred_mitigated = model_mitigated.predict(X_test_mitigated) # [INSERT CODE HERE]

# Evaluate testing set predictions using evaluate_model()
results_mitigated = evaluate_model(y_test, y_pred_mitigated, group_labels_test)

# Save your results to all_mitigated_results
results_mitigated['teammate'] = 'Teammate 1'
results_mitigated['experiment_type'] = 'post-mitigation'
results_mitigated['predictor_model'] = 'Logistic Regression' #[INSERT MODEL NAME HERE]
results_mitigated['mitigation_strategy'] = 'preprocessing' #[INSERT STRATEGY TYPE HERE: 'preprocessing', 'inprocessing', 'postprocessing']
all_mitigated_results.append(results_mitigated)

pprint(results_mitigated)

Dropping sensitive columns: ['race_Asian-Pac-Islander', 'race_Black', 'race_Other', 'race_White', 'sex_Male']


C:\Users\leona\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


{'by_group':                            accuracy  f1_score  selection_rate  \
sensitive_feature_0                                             
Female_Amer-Indian-Eskimo  0.925000  0.769231        0.125000   
Female_Asian-Pac-Islander  0.866667  0.444444        0.146667   
Female_Black               0.951002  0.560000        0.048998   
Female_Other               1.000000  1.000000        0.090909   
Female_White               0.913062  0.597303        0.089018   
Male_Amer-Indian-Eskimo    0.862069  0.600000        0.172414   
Male_Asian-Pac-Islander    0.728261  0.642857        0.407609   
Male_Black                 0.864055  0.569343        0.124424   
Male_Other                 0.857143  0.625000        0.142857   
Male_White                 0.804384  0.671492        0.261570   

                           true_positive_rate  false_positive_rate  
sensitive_feature_0                                                 
Female_Amer-Indian-Eskimo            0.625000             0.000000  

### Teammate 1's Conclusions
[Briefly describe findings and conclusions here. Compare post-mitigation results with baseline results for your model. What is the % improvement in performance post-mitigation?  ]

I found that predictive performance went down slightly but the mitigation strategy helped fairness a little. Accuracy decreased by about 0.39% post-mitigation and F1 score decreased by about 1.96% post-mitigation. So while removing sex and race did reduce some of the model's direct use of protected demographic attributes, bias still remains because other variables like relationship, occupation, marital-status, hours-per-week, and education can be used as indirect proxies for sex and race. This validates the concern that more bias mitigation should be applied in addition to simply removing the protected attributes.



## Teammate 2

In [23]:
from imblearn.over_sampling import SMOTE
# Implement your bias mitigation strategy
## If you chose preprocessing, you will train a new version of your predictor model with new/modified inputs.
## If you chose inprocessing, you will train a new version of your predictor with modified learning objective (loss function).
## If you chose postprocessing, you will implement strategies to modify the predictions (y_pred) of the trained baseline predictor model from the previous milestone without training any new version of the predictor model.

# Preprocessing: SMOTE oversampling to balance training data
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

model_t2_mitigated = RandomForestClassifier(n_estimators=100, random_state=42)
model_t2_mitigated.fit(X_train_resampled, y_train_resampled)

# Make predictions on the testing set and store them in y_pred_mitigate
y_pred_mitigated = model_t2_mitigated.predict(X_test)

# Evaluate testing set predictions using evaluate_model()
results_mitigated = evaluate_model(y_test, y_pred_mitigated, group_labels_test)

# Save your results to all_mitigated_results
results_mitigated['teammate'] = 'Teammate 2'
results_mitigated['experiment_type'] = 'post-mitigation'
results_mitigated['predictor_model'] = 'Random Forest'
results_mitigated['mitigation_strategy'] = 'preprocessing'
all_mitigated_results.append(results_mitigated)

print(results_mitigated)

ModuleNotFoundError: No module named 'imblearn'

### Teammate 2's Conclusions
[Briefly describe findings and conclusions here. Compare post-mitigation results with baseline results for your model. What is the % improvement in performance post-mitigation?]

The baseline Random Forest model achieved 85.4% accuracy and an F1 score of 0.689, but showed clear bias across groups. The equal opportunity difference was 0.700 and equalized odds difference was 0.700, meaning the model was much better at correctly identifying high earners in some groups than others, particularly disadvantaging female and minority groups.
After applying SMOTE oversampling, accuracy stayed nearly the same at 84.9% and F1 improved slightly to 0.691. The biggest improvement was in equal opportunity and equalized odds, both dropping from 0.700 to 0.500 which is a 28.6% improvement. This means the model became more consistent at correctly predicting high income across groups. However, demographic parity difference actually got slightly worse (0.299 to 0.324), so SMOTE helped with true positive rate fairness but did not reduce the overall selection rate gap between groups.
Overall, SMOTE was a kind of a success because it improved equal opportunity meaningfully without sacrificing accuracy, but further mitigation would be needed to fully close the demographic parity gap.

## Teammate 3

In [24]:
from fairlearn.postprocessing import ThresholdOptimizer

# Postprocessing: ThresholdOptimizer applies group-specific decision thresholds
# to the already-trained baseline model without retraining it
threshold_optimizer = ThresholdOptimizer(
    estimator=model_t3,
    constraints="equalized_odds",
    predict_method="predict_proba",
    objective="balanced_accuracy_score"
)

# Fit on training data to learn optimal group-specific thresholds
threshold_optimizer.fit(X_train, y_train, sensitive_features=group_labels_train)

# Make predictions on the testing set and store them in y_pred_mitigated
y_pred_mitigated = threshold_optimizer.predict(X_test, sensitive_features=group_labels_test)

# Evaluate testing set predictions using evaluate_model()
results_mitigated = evaluate_model(y_test, y_pred_mitigated, group_labels_test)

# Save your results to all_mitigated_results
results_mitigated['teammate'] = 'Teammate 3'
results_mitigated['experiment_type'] = 'post-mitigation'
results_mitigated['predictor_model'] = 'Gradient Boosting'
results_mitigated['mitigation_strategy'] = 'postprocessing'
all_mitigated_results.append(results_mitigated)

print(results_mitigated)


{'overall': {'accuracy': 0.7950248756218905, 'f1_score': 0.6630316248636859}, 'by_group':                            accuracy  f1_score  selection_rate  \
sensitive_feature_0                                             
Female_Amer-Indian-Eskimo  0.825000  0.666667        0.325000   
Female_Asian-Pac-Islander  0.760000  0.400000        0.306667   
Female_Black               0.772829  0.291667        0.258352   
Female_Other               0.954545  0.800000        0.136364   
Female_White               0.787022  0.476483        0.279950   
Male_Amer-Indian-Eskimo    0.862069  0.692308        0.275862   
Male_Asian-Pac-Islander    0.760870  0.690141        0.418478   
Male_Black                 0.776498  0.576419        0.336406   
Male_Other                 0.761905  0.615385        0.380952   
Male_White                 0.802323  0.729557        0.397040   

                           true_positive_rate  false_positive_rate  
sensitive_feature_0                                         

The baseline Gradient Boosting model reached 86.4% accuracy with an F1 score of 0.693, but revealed significant disparities across demographic groups. The equal opportunity difference and equalized odds difference were both approximately 0.700, indicating the model was far more effective at identifying high earners in some groups than others, with female and minority groups being the most disadvantaged.

After applying ThresholdOptimizer postprocessing, accuracy dropped to 79.9% and F1 decreased slightly to 0.668. However, the fairness metrics improved significantly with equal opportunity difference falling from 0.700 to 0.300, and equalized odds difference saw the same 57.1% reduction. Demographic parity difference also improved significantly from 0.281 to 0.165. True positive rates across groups became notably more consistent, narrowing from a range of 0.300 to 1.000 in the baseline down to 0.678–1.000 post-mitigation, meaning the model became far more efficient at correctly identifying high earners regardless of group membership.

Overall, ThresholdOptimizer was a success in reducing bias. The trade-off was a noticeable drop in overall accuracy, reflecting the fairness-accuracy tradeoff. Despite this drop in accuracy, for a dataset where demographic equity is a priority, the 57.1% improvement in equalized odds justifies the reduction in accuracy.

# Conclusions
_(new in this milestone)_


In [25]:
# Collect all the results in one table.
overall_results = pd.concat([pd.DataFrame(all_baseline_results), pd.DataFrame(all_mitigated_results)])
overall_results ## Note: The table displayed below in this starter notebook is for your reference, your team's table will be slightly different (e.g. different metrics, no.of sensitive attribute-based groups, actual values, etc.) upon successful completion of this notebook.

,overall,by_group,demographic_parity_difference,disparate_impact_ratio,equal_opportunity_difference,equalized_odds_difference,teammate,experiment_type,predictor_model,mitigation_strategy
0,"{'accuracy': 0.843338861249309, 'f1_score': 0....",accuracy f1_score ...,0.411712,0.119128,0.518072,0.518072,Teammate 1,baseline,Logistic Regression,NONE
1,"{'accuracy': 0.8543946932006633, 'f1_score': 0...",accuracy f1_score ...,0.298828,0.140869,0.700000,0.700000,Teammate 2,baseline,Random Forest,NONE
2,"{'accuracy': 0.8635710337202874, 'f1_score': 0...",accuracy f1_score ...,0.280563,0.125024,0.700000,0.700000,Teammate 3,baseline,Gradient Boosting,NONE
0,"{'accuracy': 0.8440022111663903, 'f1_score': 0...",accuracy f1_score ...,0.358611,0.120208,0.530120,0.530120,Teammate 1,post-mitigation,Logistic Regression,preprocessing
1,"{'accuracy': 0.7950248756218905, 'f1_score': 0...",accuracy f1_score ...,0.282115,0.325856,0.250000,0.250000,Teammate 3,post-mitigation,Gradient Boosting,postprocessing


Across all three mitigation strategies, the results revealed a consistent trade off between fairness metrics and overall performance, with each approach offering a different balance between the two.

Teammate 1's preprocessing strategy of dropping sensitive columns produced the least improvement overall. Accuracy dipped slightly and fairness metrics only improved marginally, meaning that simply removing sex and race from the feature set was not enough to meaningfully reduce bias. Variables like occupation, marital status, and hours-per-week continued to carry demographic information indirectly, limiting how much the mitigation could actually accomplish.

Teammate 2's SMOTE resampling proved to be a stronger approach, delivering a better improvement in equalized odds difference with basically no accuracy trade off. This made it the most balanced strategy of the three, improving how consistently the model identified high earners across groups without meaningfully sacrificing overall predictive performance. However, demographic parity difference actually worsened slightly, suggesting SMOTE helped with true positive rate fairness but did not fully help with the selection rate gap between groups.

Teammate 3's ThresholdOptimizer postprocessing achieved the greatest fairness improvements of all three strategies, with equalized odds difference and demographic parity difference both dropping substantially. The trade off was a noticeable decline in overall accuracy, reflecting the unavoidable tradeoff between fairness and performance. 

Overall, postprocessing proved to be the most effective approach at directly improving our fairness metrics, while preprocessing strategies served as a better balance by  better preserving accuracy but delivered less fairness improvements. No single strategy eliminated bias entirely, pointing to the need for combining multiple mitigation techniques in practice.

# References

[List the references you used to complete this milestone here.]
- Teammate 1: The UCI Machine Learning Repository Adult dataset page was used to import the dataset directly in Python using the ucimlrepo package and fetch_ucirepo(id=2). Becker, B., & Kohavi, R. (1996). Adult [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5XW20
Evaluation metric and Logistic Regression code was adapted from prior course discussion notebooks, especially Week 4’s Fairlearn metric and Week 3's Logistic Regression examples. The code was modified for the Adult dataset and our selected sensitive feature groups.
- Teammate 2: SMOTE oversampling was implemented using the imbalanced-learn library. Random Forest classifier was used from scikit-learn. Fairness evaluation code was adapted from prior course discussion notebooks like Week 3 and Week 4's Fairlearn metric examples.
- Teammate 3: Gradient Boosting classifier was used from scikit-learn. ThresholdOptimizer postprocessing for bias mitigation was implemented using the fairlearn library.

# Disclosures

[Disclose use of generative AI and similar tools here.]
- Teammate 1: ChatGPT was used for clarification of assignment concepts and wording feedback, but not to generate final code or complete the assignment.
- Teammate 2: ChatGPT used for clarification of assignment concepts and needed tasks and for wording feedback on conclusion section. Was not used for final code generation or to complete the whole project.
- Teammate 3: Claude was used for clarification of assingment concepts and for helping format the conclusion. Was not used for final code generation or to complete the whole project.